In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import glob
import os.path
import datetime
import os
from math import atan2, degrees, sin, cos, sqrt, radians
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import warnings
import pickle

warnings.filterwarnings('ignore')


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# ---------------------------------------------------
# Funções para leitura dos arquivos .plt e labels
# ---------------------------------------------------

def read_plt(plt_file):
    points = pd.read_csv(plt_file, skiprows=6, header=None,
                         parse_dates=[[5, 6]], infer_datetime_format=True)
    points.rename(inplace=True, columns={'5_6': 'time', 0: 'lat', 1: 'lon', 3: 'alt'})
    points.drop(inplace=True, columns=[2, 4])
    return points

mode_names = ['walk', 'bike', 'bus', 'car', 'subway','train', 'airplane', 'boat', 'run', 'motorcycle', 'taxi']
mode_ids = {s : i + 1 for i, s in enumerate(mode_names)}

def read_labels(labels_file):
    labels = pd.read_csv(labels_file, skiprows=1, header=None,
                         parse_dates=[[0, 1], [2, 3]],
                         infer_datetime_format=True, delim_whitespace=True)
    labels.columns = ['start_time', 'end_time', 'label']
    labels['label'] = [mode_ids[i] if i in mode_ids else 0 for i in labels['label']]
    return labels

def apply_labels(points, labels):
    indices = labels['start_time'].searchsorted(points['time'], side='right') - 1
    no_label = (indices < 0) | (points['time'].values >= labels['end_time'].iloc[indices].values)
    points['label'] = labels['label'].iloc[indices].values
    points.loc[no_label, 'label'] = 0

# ---------------------------------------------------
# read_user() com proteção concat vazio
# ---------------------------------------------------
def read_user(user_folder):
    labels = None
    plt_files = glob.glob(os.path.join(user_folder, 'Trajectory', '*.plt'))
    if len(plt_files) == 0:
        print("Nenhum arquivo .plt encontrado em:", user_folder)
        return None
    df = pd.concat([read_plt(f) for f in plt_files], ignore_index=True)
    labels_file = os.path.join(user_folder, 'labels.txt')
    if os.path.exists(labels_file):
        labels = read_labels(labels_file)
        apply_labels(df, labels)
    else:
        df['label'] = 0
    return df

# ---------------------------------------------------
# read_all_users() com proteção pastas sem arquivos
# ---------------------------------------------------
def read_all_users(folder):
    subfolders = [f for f in os.listdir(folder) if os.path.isdir(os.path.join(folder,f))]
    dfs = []
    for i, sf in enumerate(subfolders):
        print('[%d/%d] processing user %s' % (i + 1, len(subfolders), sf))
        df_user = read_user(os.path.join(folder,sf))
        if df_user is not None:
            df_user['user'] = int(sf)
            dfs.append(df_user)
    if len(dfs) == 0:
        raise ValueError("Nenhum usuário com arquivos .plt encontrado!")
    return pd.concat(dfs, ignore_index=True)

# ---------------------------------------------------
# Leitura de todos os usuários
# ---------------------------------------------------
df = read_all_users(r"../dados/geolife/")

# ---------------------------------------------------
# Pré-processamento
# ---------------------------------------------------
df['time'] = pd.to_datetime(df['time'])
df['time_diff'] = df['time'].diff().dt.total_seconds()

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

df['distance'] = calculate_distance(df['lat'].shift(), df['lon'].shift(), df['lat'], df['lon'])
df['speed'] = df['distance'] / df['time_diff']
df['acceleration'] = df['speed'].diff() / df['time_diff']

def calculate_bearing(lat1, lon1, lat2, lon2):
    delta_lon = np.radians(lon2 - lon1)
    lat1, lat2 = np.radians(lat1), np.radians(lat2)
    y = np.sin(delta_lon) * np.cos(lat2)
    x = np.cos(lat1)*np.sin(lat2) - np.sin(lat1)*np.cos(lat2)*np.cos(delta_lon)
    bearing = (np.degrees(np.arctan2(y, x)) + 360) % 360
    return bearing

df['bearing'] = calculate_bearing(df['lat'].shift(), df['lon'].shift(), df['lat'], df['lon'])
df = df.dropna()
df['month'] = df['time'].dt.month
df['day'] = df['time'].dt.day
df['hour'] = df['time'].dt.hour

# ---------------------------------------------------
# Seleção de linhas por label
# ---------------------------------------------------
num_rows = 10000000
df_subset = df.head(num_rows)
df_subset = df_subset.drop(['lat', 'lon', 'user'], axis=1)
df_subset = df_subset[np.isfinite(df_subset).all(1)]
df_subset['time'] = pd.to_datetime(df_subset['time'])
df_subset['month'] = df_subset['time'].dt.month
df_subset['day'] = df_subset['time'].dt.day
df_subset['hour'] = df_subset['time'].dt.hour

label_0_count = 10000
label_8_count = 1985
label_7_count = 734
label_9_count = 276

sorted_df = df_subset.sort_values(by=['label', 'time'], ascending=[True, True])
grouped_df = sorted_df.groupby('label')
selected_dfs = []

for label, group in grouped_df:
    if label == 0:
        selected_group = group.head(label_0_count)
    elif label == 1:
        selected_group = group.head(label_0_count)
    elif label == 3:
        selected_group = group.head(label_0_count)
    elif label == 2:
        selected_group = group.head(label_0_count)
    elif label == 4:
        selected_group = group.head(label_0_count)
    elif label == 5:
        selected_group = group.head(label_0_count)
    elif label == 6:
        selected_group = group.head(label_0_count)
    elif label == 7:
        selected_group = group.head(label_7_count)
    elif label == 8:
        selected_group = group.head(label_8_count)
    elif label == 9:
        selected_group = group.head(label_9_count)
    elif label == 11:
        selected_group = group.head(label_0_count)
    else:
        continue
    selected_dfs.append(selected_group)

selected_df = pd.concat(selected_dfs, ignore_index=True)
print(selected_df['label'].value_counts())

# ---------------------------------------------------
# Proteção contra inf/NaN antes do treino
# ---------------------------------------------------
selected_df['speed'] = selected_df['speed'].replace([np.inf, -np.inf], np.nan)
selected_df['acceleration'] = selected_df['acceleration'].replace([np.inf, -np.inf], np.nan)
selected_df = selected_df.dropna(subset=['speed','acceleration'])

# ---------------------------------------------------
# Random Forest
# ---------------------------------------------------
X = selected_df.drop(['label', 'time'], axis=1)
print(X.columns)
y = selected_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_classifier = RandomForestClassifier()
rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)
print("Accuracy:", rf_classifier.score(X_test, y_test))
print(classification_report(y_test, y_pred))

# ---------------------------------------------------
# Teste com novos dados e predição do veículo
# ---------------------------------------------------
testdata = {
    'time': ['9/30/2008 22:23', '9/30/2008 22:24', '9/30/2008 22:25', '9/30/2008 22:26'],
    'lat': [35.661356, 35.661113, 35.660988, 35.660869],
    'lon': [94.06079, 94.061095, 94.061243, 94.061403],
    'alt': [0,0,0,0],
    'label': [11,11,11,11]
}

testtest = pd.DataFrame(testdata)

testtest['time'] = pd.to_datetime(testtest['time'])
testtest['time_diff'] = (testtest['time'] - testtest['time'].shift()).dt.total_seconds()
testtest['distance'] = calculate_distance(testtest['lat'].shift().fillna(0),
                                         testtest['lon'].shift().fillna(0),
                                         testtest['lat'], testtest['lon'])
testtest['speed'] = testtest['distance'] / testtest['time_diff']
testtest['acceleration'] = testtest['speed'].diff()
testtest['bearing'] = calculate_bearing(testtest['lat'].shift().fillna(0),
                                       testtest['lon'].shift().fillna(0),
                                       testtest['lat'], testtest['lon'])
testtest['month'] = testtest['time'].dt.month
testtest['day'] = testtest['time'].dt.day
testtest['hour'] = testtest['time'].dt.hour

testtest['speed'] = testtest['speed'].replace([np.inf, -np.inf], np.nan)
testtest['acceleration'] = testtest['acceleration'].replace([np.inf, -np.inf], np.nan)
testtest = testtest.dropna(subset=['speed','acceleration'])

X_new = testtest.drop(['label','time','lat','lon'], axis=1)
y_pred_new = rf_classifier.predict(X_new)
print("Predicted labels (estimativa de veículo):", y_pred_new)


# Save
with open("random_forest_model.pkl", "wb") as f:
    pickle.dump(rf_classifier, f)

[1/183] processing user 033
[2/183] processing user 170
[3/183] processing user 023
[4/183] processing user 147
[5/183] processing user 024
[6/183] processing user 050
[7/183] processing user 106
[8/183] processing user 015
[9/183] processing user 041
[10/183] processing user 181
[11/183] processing user 142
[12/183] processing user 038
[13/183] processing user 051
[14/183] processing user 058
[15/183] processing user 052
[16/183] processing user 137
[17/183] processing user 139
[18/183] processing user 064
[19/183] processing user 009
[20/183] processing user 124
[21/183] processing user 089
[22/183] processing user 020
[23/183] processing user 042
[24/183] processing user 158
[25/183] processing user 090
[26/183] processing user 129
[27/183] processing user 026
[28/183] processing user 161
[29/183] processing user 063
[30/183] processing user 036
[31/183] processing user 010
[32/183] processing user 146
[33/183] processing user 154
[34/183] processing user 013
[35/183] processing use

In [3]:
df

,time,lat,lon,alt,label,user,time_diff,distance,speed,acceleration,bearing,month,day,hour
2,2009-01-11 19:47:09,39.951948,116.361209,149.0,0,33,4.0,0.018585,0.004646,-0.001858,145.656794,1,11,19
3,2009-01-11 19:47:10,39.951866,116.361290,151.0,0,33,1.0,0.011437,0.011437,0.006791,142.865689,1,11,19
4,2009-01-11 19:47:13,39.951777,116.361377,151.0,0,33,3.0,0.012367,0.004122,-0.002438,143.153621,1,11,19
5,2009-01-11 19:47:15,39.951669,116.361505,152.0,0,33,2.0,0.016225,0.008113,0.001995,137.743365,1,11,19
6,2009-01-11 19:47:16,39.951555,116.361631,154.0,0,33,1.0,0.016614,0.016614,0.008502,139.726007,1,11,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24876973,2009-07-09 13:34:35,39.995370,116.393870,112.0,0,156,5.0,0.008450,0.001690,0.000078,350.717990,7,9,13
24876974,2009-07-09 13:34:40,39.995405,116.393843,116.0,0,156,5.0,0.004521,0.000904,-0.000157,329.417410,7,9,13
24876975,2009-07-09 13:34:45,39.995504,116.393877,109.0,0,156,5.0,0.011383,0.002277,0.000274,14.740636,7,9,13
24876976,2009-07-09 13:34:50,39.995582,116.393899,98.0,0,156,5.0,0.008873,0.001775,-0.000100,12.192892,7,9,13


In [5]:
df["label"].value_counts()

label
0     19315534
1      1367247
3      1207317
2       804129
6       560967
4       505898
5       257396
11      217134
7         9183
8         3559
9         1971
10         338
Name: count, dtype: int64

In [4]:
testdata = {
    'time': ['9/30/2008 22:23', '9/30/2008 22:24', '9/30/2008 22:25', '9/30/2008 22:26'],
    'lat': [35.661356, 35.661113, 35.660988, 35.660869],
    'lon': [94.06079, 94.061095, 94.061243, 94.061403],
    'alt': [0,0,0,0],
    'label': [11,11,11,11]
}

testtest = pd.DataFrame(testdata)

testtest['time'] = pd.to_datetime(testtest['time'])
testtest['time_diff'] = (testtest['time'] - testtest['time'].shift()).dt.total_seconds()
testtest['distance'] = calculate_distance(testtest['lat'].shift().fillna(0),
                                         testtest['lon'].shift().fillna(0),
                                         testtest['lat'], testtest['lon'])
testtest['speed'] = testtest['distance'] / testtest['time_diff']
testtest['acceleration'] = testtest['speed'].diff()
testtest['bearing'] = None
calculate_bearing(testtest['lat'].shift().fillna(0),
                                       testtest['lon'].shift().fillna(0),
                                       testtest['lat'], testtest['lon'])
testtest['month'] = testtest['time'].dt.month
testtest['day'] = testtest['time'].dt.day
testtest['hour'] = testtest['time'].dt.hour

testtest['speed'] = testtest['speed'].replace([np.inf, -np.inf], np.nan)
testtest['acceleration'] = testtest['acceleration'].replace([np.inf, -np.inf], np.nan)
testtest = testtest.dropna(subset=['speed','acceleration'])

X_new = testtest.drop(['label','time','lat','lon'], axis=1)
y_pred_new = rf_classifier.predict(X_new)
print("Predicted labels (estimativa de veículo):", y_pred_new)


# Save
with open("random_forest_model.pkl", "wb") as f:
    pickle.dump(rf_classifier, f)

Predicted labels (estimativa de veículo): [0 0]
